# PA3 — run everything (Colab + Google Drive)
1. Upload **`atml-pa3.zip`**, the supplied **`first_plot_questions.yaml`** and the supplied **`ATML-PA3-Data/`** folder to Drive.
2. Runtime → Change runtime type → **GPU (A100 or L4 recommended; T4 works but is slow and tight on memory for Full FT)**.
3. Edit the paths in the next cell, then run cells top to bottom. Every stage is **resumable**: if Colab disconnects,
   re-run the setup cells and then the cell that was interrupted — it continues from the last checkpoint.

The repo is unzipped **onto Drive**, so `results/` (small, commit to GitHub) survives resets; big artifacts go to `PA3_OUT`.

In [ ]:
# ======== EDIT THESE ========
DRIVE_ROOT = '/content/drive/MyDrive/ATML_PA3'           # folder holding the uploads below
ZIP_PATH   = f'{DRIVE_ROOT}/atml-pa3.zip'
REPO_DIR   = f'{DRIVE_ROOT}/atml-pa3'                    # repo lives on Drive (results persist)
PA3_OUT    = f'{DRIVE_ROOT}/outputs'                     # checkpoints, merged models, features, raw samples
QUAL_YAML  = f'{DRIVE_ROOT}/first_plot_questions.yaml'   # supplied Task-1 prompts
T3_DATA    = f'{DRIVE_ROOT}/ATML-PA3-Data'               # supplied Task-3 rollouts (folder with README.MD)
LIVE_REVISION = 'aa8e72537993ba99e69dfaafa59ed015b17504d1'   # Qwen2.5-3B-Instruct revision pinned in ATML-PA3-Data/README.MD
# ============================
from google.colab import drive
drive.mount('/content/drive')
import os, zipfile, shutil
if not os.path.exists(f'{REPO_DIR}/README.md'):
    with zipfile.ZipFile(ZIP_PATH) as z: z.extractall(DRIVE_ROOT)
os.makedirs(f'{REPO_DIR}/data', exist_ok=True)
if os.path.exists(QUAL_YAML): shutil.copy(QUAL_YAML, f'{REPO_DIR}/data/first_plot_questions.yaml')
os.environ['PA3_OUT'] = PA3_OUT
os.environ['HF_HOME'] = '/content/hf_cache'              # local, fast; re-downloads after a reset
os.environ['HF_DATASETS_TRUST_REMOTE_CODE'] = '1'        # some lm-eval datasets still use loading scripts
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.makedirs(PA3_OUT, exist_ok=True)
%cd {REPO_DIR}

In [ ]:
!pip -q install -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python -m common.utils --record_env results/env_colab.json

## Task 1 — LLM fine-tuning (Qwen3-0.6B on MetaMathQA)
Data selection (first 40k eligible, decontaminated), activation-probe set (seed 6304, sampled before any results), no-thinking tokenizer.

In [ ]:
!python -m task1_llm_finetune.data

Train the five variants (each 1,250 updates; auto-resumes). Run one cell per session if your runtime is time-limited.

In [ ]:
!python -m task1_llm_finetune.train --variant full_ft

In [ ]:
!python -m task1_llm_finetune.train --variant l2sp

In [ ]:
!python -m task1_llm_finetune.train --variant lora4

In [ ]:
!python -m task1_llm_finetune.train --variant lora16

In [ ]:
!python -m task1_llm_finetune.train --variant oplora16

Evaluate the seven conditions with lm-evaluation-harness (zero-shot, FP16, complete splits). Adapters are merged into fresh backbones with a merged-vs-unmerged logit check.
On a T4, lower the GSM8K batch size: append `--set eval.gen_batch_size=8`.

In [ ]:
for cond in ['baseline','full_ft_mid','full_ft','l2sp','lora4','lora16','oplora16']:
    !python -m task1_llm_finetune.evaluate --condition {cond}

In [ ]:
!python -m task1_llm_finetune.diagnostics
!python -m task1_llm_finetune.qualitative
!python -m task1_llm_finetune.analyze

After this, open `results/task1/qualitative/annotation_sheet_42.csv` and fill the `ANN_*` columns **yourself** (the report's annotation must be your own).

## Task 2 — SigLIP–Qwen VLM and visual grounding
Downloads COCO-2014 annotations + VQAv2 (≈0.5 GB) and only the ≈11k images actually used.

In [ ]:
!python -m task2_vlm.prepare_data
!python -m task2_vlm.prepare_data --inspect   # label grids for manual checking (independent of the model)
!python -m task2_vlm.cache_features

In [ ]:
# copy the feature cache to local disk for fast random access during training
import glob
os.makedirs('/content/feat_local', exist_ok=True)
for f in glob.glob(f'{PA3_OUT}/data/task2/features/siglip_feats_fp16.npy'):
    if not os.path.exists('/content/feat_local/' + os.path.basename(f)): shutil.copy(f, '/content/feat_local/')
os.environ['PA3_FEAT_LOCAL'] = '/content/feat_local'

In [ ]:
!python -m task2_vlm.train

In [ ]:
!python -m task2_vlm.evaluate
!python -m task2_vlm.analyze

**Optional** 49-token compression run (new connector + adapters, same seed/data/budgets/evaluation).

In [ ]:
# !python -m task2_vlm.train --set num_visual_tokens=49 run_name=pool49
# !python -m task2_vlm.evaluate --set num_visual_tokens=49 run_name=pool49
# !python -m task2_vlm.analyze --set num_visual_tokens=49 run_name=pool49

## Task 3 — Test-time compute
Offline parts are CPU-only (the MATH answer grouping takes ~5 min). Folder and column names in `configs/task3.yaml` are already set to the supplied export. MathVista metadata (question/answer type, precision, choices) is downloaded from `AI4Math/MathVista` testmini. Check `results/task3/checks/*_alignment_and_scorer.json`: capped / null-extraction / token totals must equal the README table (MATH 523 / 521 / 20,742,094; MathVista 487 / 3,032 / 8,961,703).

In [ ]:
!ls -R "{T3_DATA}"

In [ ]:
!python -m task3_ttc.data --set data_root="{T3_DATA}"
!python -m task3_ttc.offline_eval --set data_root="{T3_DATA}"
!python -m task3_ttc.signals --set data_root="{T3_DATA}"
!python -m task3_ttc.adaptive --set data_root="{T3_DATA}"

Live generation on the fixed 50 MATH questions (GPU; resumable — every call is appended to a JSONL).

In [ ]:
assert LIVE_REVISION, 'set LIVE_REVISION in the first cell (from the supplied README.MD)'
!python -m task3_ttc.live_generation --set data_root="{T3_DATA}" live.revision="'{LIVE_REVISION}'"
!python -m task3_ttc.analyze --set data_root="{T3_DATA}"

## Wrap-up
* `results/` now holds every table/figure/JSON used in the report — commit the repo (without `outputs/`/`data/`) to a **public** GitHub repo.
* Record the final environment too.

In [ ]:
!python -m common.utils --record_env results/env_final.json
!du -sh results; find results -maxdepth 3 -type d | sort | head -80